### Geldtransporter – Lösung Schritt für Schritt

*31. Bundeswettbewerb Informatik (2012), 1. Runde, Aufgabe 2*

Zum Transport von Geld werden schwer bewachte Geldtransporter eingesetzt. In einem solchen Geldtransporter können Koffer mit Münzen transportiert werden. Die Koffer enthalten Münzen in unterschiedlicher Menge und mit unterschiedlichem Gesamtwert. Entsprechend unterscheiden sich die Koffer in Gewicht und Wert.

Da kommt so einiges an Gewicht zusammen. Es ist daher notwendig, den Transporter gleichmäßig zu beladen, so dass er nicht zu einer Seite umkippen kann.

Unser Transporter hat links und rechts je einen Kofferraum. Für jeden Kofferraum lassen sich Gesamtwert und Gesamtgewicht der darin enthaltenen Koffer bestimmen. Damit der Transporter keine Schlagseite bekommt, müssen die Koffer so eingeräumt werden, dass die Differenz zwischen den beiden Gesamtgewichten minimal ist. Aus versicherungstechnischen Gründen dürfen die beiden Gesamtwerte sich außerdem um höchstens 10.000 Euro unterscheiden.

#### Aufgabe 2

Schreibe ein Programm zur Verteilung der Geldkoffer auf die beiden Kofferräume. Überprüfe dein Programm mit den auf www.bundeswettbewerb-informatik.de abgelegten Beispielen mit Angaben zu Werten und Gewichten der einzelnen Koffer.

**Zu den Daten:** In `beispieldaten/` liegen `eingabe15.txt`, `eingabe20.txt`, `eingabe30.txt` und `eingabe40.txt`. Eine Formatbeschreibung gibt es nicht. Die erste Zeile ist die Anzahl der Koffer, dann folgt je Koffer eine Zeile mit zwei Zahlen. Aus den Zahlen schließen wir: erst der **Wert** (in Euro, immer ein Vielfaches von 1 000, zwischen 7 000 und 944 000), dann das **Gewicht** (ganze Zahlen, bei 15 und 20 Koffern bis 82, bei 30 und 40 Koffern bis 3 512). Die Deutung passt zu den Ergebnissen der Lösungshinweise (die optimalen Gewichtsunterschiede 5, 11, 112 und 5 finden wir genau so).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Brute Force:** Alle Möglichkeiten ausprobieren. Bei `n` Koffern gibt es `2^n` Verteilungen (jeder Koffer links oder rechts). Das geht für 15 Koffer gut, für 40 Koffer nicht mehr (mehr als 5 · 10^11 Möglichkeiten).
- **Rekursion mit Backtracking:** Man verteilt die Koffer einen nach dem anderen und macht bei jedem Koffer zwei Versuche (links, rechts).
- **Branch and Bound („verzweigen und beschränken“):** Man bricht einen Versuch **früh ab**, sobald klar ist, dass er nicht mehr zu einer (besseren) Lösung führen kann. Dafür braucht man **Schranken**: Was kann mit den restlichen Koffern höchstens noch ausgeglichen werden?
- **Zustände zusammenfassen (Dynamische Programmierung):** Verschiedene Teilverteilungen mit demselben Wert- und Gewichtsunterschied sind für den Rest gleich. Man merkt sich jeden Zustand nur einmal (für Neugierige).
- **Das Partitionsproblem:** Die Aufgabe ist eine Variante eines berühmten schweren Problems. Ein schnelles Verfahren für *alle* Eingaben ist nicht bekannt.

**Nützliche Funktionen und Bibliotheken**
- `itertools.product([0, 1], repeat=k)`: alle Folgen aus `k` Nullen und Einsen.
- `sorted(liste, reverse=True)`: sortiert absteigend. Bei Paaren (Tupeln) wird nach dem ersten Eintrag sortiert, bei Gleichheit nach dem zweiten.
- `abs(x)`: Betrag.
- `dict` (Wörterbuch) zum Zählen und Merken von Zuständen.

## Die Idee in Worten

Jeder Koffer kommt links (0) oder rechts (1). Wir schreiben die Entscheidung als Folge aus 0 und 1, eine Zahl pro Koffer. Für eine solche Verteilung berechnen wir

- den **Wertunterschied** `|Wert links − Wert rechts|`: er muss höchstens 10 000 sein (dann nennen wir die Verteilung **zulässig**),
- den **Gewichtsunterschied** `|Gewicht links − Gewicht rechts|`: er soll unter allen zulässigen Verteilungen **möglichst klein** sein.

**Zwei Vereinfachungen**
1. Vertauscht man links und rechts, ändert sich kein Unterschied. Wir legen daher den **ersten Koffer immer nach links** und haben nur `2^(n−1)` statt `2^n` Verteilungen (so zählen auch die Lösungshinweise).
2. Statt beide Summen zu verwalten, genügt der **Unterschied**: Beim Verteilen rechnen wir `dwert = Wert links − Wert rechts` und `dgew = Gewicht links − Gewicht rechts`. Ein Koffer links erhöht beide Unterschiede, ein Koffer rechts verringert sie.

**Die Schranken für Branch and Bound.** Angenommen, wir haben schon einige Koffer verteilt und die restlichen Koffer haben zusammen den Wert `R`.
- Jeder weitere Koffer ändert `dwert` um seinen Wert nach oben oder unten. Selbst wenn alle restlichen Koffer in die richtige Richtung gehen, schaffen sie höchstens `R`. Ist `|dwert| − R > 10 000`, kann **keine** zulässige Verteilung mehr entstehen: Abbruch.
- Genauso mit dem Gewicht: Haben die restlichen Koffer zusammen das Gewicht `G` und gilt `|dgew| − G ≥ beste`, wobei `beste` der kleinste bisher gefundene Gewichtsunterschied ist, kann dieser Zweig nicht mehr **besser** werden: Abbruch.

Die Schranken wirken stärker, wenn wir die **großen Koffer zuerst** verteilen, denn dann werden die Restsummen schnell klein. Wir sortieren deshalb die Koffer nach Wert, den größten zuerst.

## Schritt 1: Daten einlesen

Ein Koffer ist ein Paar `(Wert, Gewicht)`. Die Datei wird zu einer Liste solcher Paare.

In [ ]:
def lies_koffer(dateiname):
    zeilen = open(dateiname).read().split('\n')
    anzahl = int(zeilen[0])
    koffer = []
    for zeile in zeilen[1:1 + anzahl]:
        teile = zeile.split()
        koffer.append((int(teile[0]), int(teile[1])))      # (Wert, Gewicht)
    return koffer


koffer15 = lies_koffer('beispieldaten/eingabe15.txt')
print(len(koffer15), 'Koffer')
print(koffer15[:4], '...')

15 Koffer
[(96000, 27), (126000, 21), (115000, 27), (125000, 15)] ...


## Schritt 2: Eine Verteilung bewerten

Eine Verteilung ist eine Liste `seiten` mit einer 0 (links) oder 1 (rechts) pro Koffer. Die Funktion `summen` berechnet Wert und Gewicht beider Seiten, `ist_zulaessig` prüft die Versicherungsbedingung, `zeige` druckt alles übersichtlich (so verlangen es die Bewertungskriterien: Gesamtwerte, Gesamtgewichte und Differenzen).

In [ ]:
def summen(koffer, seiten):
    wert = [0, 0]                       # wert[0] links, wert[1] rechts
    gewicht = [0, 0]
    for i in range(len(koffer)):
        wert[seiten[i]] = wert[seiten[i]] + koffer[i][0]
        gewicht[seiten[i]] = gewicht[seiten[i]] + koffer[i][1]
    return wert, gewicht


def ist_zulaessig(koffer, seiten):
    wert, gewicht = summen(koffer, seiten)
    return abs(wert[0] - wert[1]) <= 10000


def zeige(koffer, seiten):
    wert, gewicht = summen(koffer, seiten)
    print('links :  Wert', wert[0], ' Gewicht', gewicht[0], ' (', seiten.count(0), 'Koffer )')
    print('rechts:  Wert', wert[1], ' Gewicht', gewicht[1], ' (', seiten.count(1), 'Koffer )')
    print('Differenz: Wert', abs(wert[0] - wert[1]), ' Gewicht', abs(gewicht[0] - gewicht[1]))


# Die ersten 7 Koffer nach links, der Rest nach rechts
probe = [0] * 7 + [1] * 8
zeige(koffer15, probe)
print('zulässig:', ist_zulaessig(koffer15, probe))

links :  Wert 820000  Gewicht 202  ( 7 Koffer )
rechts:  Wert 863000  Gewicht 253  ( 8 Koffer )
Differenz: Wert 43000  Gewicht 51
zulässig: False


Diese Verteilung ist also **nicht** zulässig: Der Wertunterschied 43 000 ist größer als 10 000. Wir suchen unter den zulässigen Verteilungen die mit dem kleinsten Gewichtsunterschied.

## Schritt 3: Brute Force für 15 Koffer

`itertools.product([0, 1], repeat=14)` liefert nacheinander alle 2^14 = 16 384 Folgen aus 0 und 1 für die Koffer 2 bis 15; der erste Koffer bleibt links. Wir bewerten jede Verteilung und merken uns

- wie viele Verteilungen zulässig sind (**Lösungen**),
- den kleinsten Gewichtsunterschied unter den Lösungen,
- wie viele Lösungen diesen kleinsten Unterschied haben (**optimale Lösungen**).

In [ ]:
import itertools


def brute_force(koffer):
    n = len(koffer)
    verteilungen = 0
    loesungen = 0
    beste = None
    optimal = 0
    beste_seiten = None
    for rest in itertools.product([0, 1], repeat=n - 1):
        seiten = [0] + list(rest)               # der erste Koffer kommt links
        verteilungen = verteilungen + 1
        wert, gewicht = summen(koffer, seiten)
        if abs(wert[0] - wert[1]) <= 10000:
            loesungen = loesungen + 1
            unterschied = abs(gewicht[0] - gewicht[1])
            if beste is None or unterschied < beste:
                beste = unterschied
                optimal = 1
                beste_seiten = seiten
            elif unterschied == beste:
                optimal = optimal + 1
    return verteilungen, loesungen, beste, optimal, beste_seiten


verteilungen, loesungen, beste, optimal, seiten15 = brute_force(koffer15)
print('Verteilungen:', verteilungen, ' Lösungen:', loesungen, ' kleinster Gewichtsunterschied:', beste, ' optimale Lösungen:', optimal)
zeige(koffer15, seiten15)

Verteilungen: 16384  Lösungen: 77  kleinster Gewichtsunterschied: 5  optimale Lösungen: 3
links :  Wert 839000  Gewicht 230  ( 8 Koffer )
rechts:  Wert 844000  Gewicht 225  ( 7 Koffer )
Differenz: Wert 5000  Gewicht 5


Das stimmt mit der Tabelle der Lösungshinweise überein: 16 384 Verteilungen, 77 Lösungen, davon 3 optimal mit Gewichtsunterschied 5.

Bei 20 Koffern wären es schon 524 288 Verteilungen, bei 30 Koffern über 500 Millionen und bei 40 Koffern über 500 Milliarden. Das ist für Brute Force zu viel.

## Schritt 4: Branch and Bound

Wir verteilen die Koffer rekursiv: `rekursion(i, dwert, dgew)` hat die Koffer vor Nummer `i` schon verteilt, `dwert` und `dgew` sind die Unterschiede (links minus rechts). Vor der Rekursion berechnen wir zwei Listen mit **Restsummen**: `rest_wert[i]` ist der Wert aller Koffer ab Nummer `i`, `rest_gew[i]` ihr Gewicht.

Am Anfang jedes Aufrufs prüfen wir die beiden Schranken aus der Idee. Danach:
- sind alle Koffer verteilt (`i == n`), haben wir eine neue beste Lösung (die Schranken haben schon sichergestellt, dass sie zulässig und besser als die bisherige ist),
- sonst probieren wir den Koffer links und rechts (der Koffer 0 nur links).

Damit die Funktion die besten Ergebnisse nach außen bringt, speichern wir sie in einem Wörterbuch `stand`, das alle Aufrufe gemeinsam verändern. Wir zählen auch die Aufrufe, um zu sehen, wie viel Arbeit gespart wird.

In [ ]:
def suche(koffer):
    # Die großen Koffer (nach Wert) zuerst: sorted sortiert Paare nach dem ersten Eintrag, hier dem Wert
    koffer = sorted(koffer, reverse=True)
    n = len(koffer)
    rest_wert = [0] * (n + 1)
    rest_gew = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        rest_wert[i] = rest_wert[i + 1] + koffer[i][0]
        rest_gew[i] = rest_gew[i + 1] + koffer[i][1]

    stand = {'beste': None, 'seiten': None, 'aufrufe': 0}
    seiten = [0] * n

    def rekursion(i, dwert, dgew):
        stand['aufrufe'] = stand['aufrufe'] + 1
        # Schranke 1: Der Wertunterschied kann nicht mehr unter 10 000 kommen
        if abs(dwert) - rest_wert[i] > 10000:
            return
        # Schranke 2: Das Gewicht kann nicht mehr besser als das beste bisherige werden
        if stand['beste'] is not None and abs(dgew) - rest_gew[i] >= stand['beste']:
            return
        if i == n:
            stand['beste'] = abs(dgew)
            stand['seiten'] = seiten[:]
            return
        wert, gewicht = koffer[i]
        seiten[i] = 0                                   # links: Unterschiede wachsen
        rekursion(i + 1, dwert + wert, dgew + gewicht)
        if i > 0:                                       # Koffer 0 bleibt links (Symmetrie)
            seiten[i] = 1                               # rechts: Unterschiede sinken
            rekursion(i + 1, dwert - wert, dgew - gewicht)

    rekursion(0, 0, 0)
    return koffer, stand['beste'], stand['seiten'], stand['aufrufe']


k, beste, seiten, aufrufe = suche(koffer15)
print('kleinster Gewichtsunterschied:', beste, ' Aufrufe:', aufrufe)
zeige(k, seiten)

kleinster Gewichtsunterschied: 5  Aufrufe: 10852
links :  Wert 844000  Gewicht 225  ( 7 Koffer )
rechts:  Wert 839000  Gewicht 230  ( 8 Koffer )
Differenz: Wert 5000  Gewicht 5


Der Gewichtsunterschied 5 stimmt mit Brute Force überein. Statt über 32 000 möglicher Aufrufe (der volle Baum hat 2^15 − 1) reichen hier knapp 11 000. Jetzt alle vier Beispiele (die Koffer erscheinen dabei sortiert, deshalb sehen die Verteilungen anders aus als eben):

In [ ]:
ergebnisse = {}
for anzahl in [15, 20, 30, 40]:
    koffer = lies_koffer('beispieldaten/eingabe' + str(anzahl) + '.txt')
    k, beste, seiten, aufrufe = suche(koffer)
    ergebnisse[anzahl] = (beste, aufrufe)
    print('=== eingabe' + str(anzahl) + '.txt:  Gewichtsunterschied', beste, ' Aufrufe', aufrufe)
    zeige(k, seiten)

=== eingabe15.txt:  Gewichtsunterschied 5  Aufrufe 10852
links :  Wert 844000  Gewicht 225  ( 7 Koffer )
rechts:  Wert 839000  Gewicht 230  ( 8 Koffer )
Differenz: Wert 5000  Gewicht 5
=== eingabe20.txt:  Gewichtsunterschied 11  Aufrufe 93302
links :  Wert 5943000  Gewicht 374  ( 10 Koffer )
rechts:  Wert 5951000  Gewicht 385  ( 10 Koffer )
Differenz: Wert 8000  Gewicht 11
=== eingabe30.txt:  Gewichtsunterschied 112  Aufrufe 1126434
links :  Wert 710000  Gewicht 3115  ( 11 Koffer )
rechts:  Wert 703000  Gewicht 3227  ( 19 Koffer )
Differenz: Wert 7000  Gewicht 112
=== eingabe40.txt:  Gewichtsunterschied 5  Aufrufe 7508348
links :  Wert 1321000  Gewicht 4934  ( 10 Koffer )
rechts:  Wert 1329000  Gewicht 4929  ( 30 Koffer )
Differenz: Wert 8000  Gewicht 5


Die optimalen Gewichtsunterschiede **5, 11, 112 und 5** sind genau die Zahlen aus der Tabelle der Lösungshinweise. Man sieht auch, wie viel Arbeit die Schranken sparen: Bei 40 Koffern sind es rund 7,5 Millionen Aufrufe statt über 10^12 beim vollständigen Durchgehen.

Dass die Verteilung wirklich zulässig und optimal ist, lässt sich in `zeige` ablesen: Der Wertunterschied bleibt unter 10 000, und der Gewichtsunterschied ist der genannte.

### Kontrolle gegen Brute Force

Damit wir `suche` trauen können, vergleichen wir sie auf vielen kleinen Zufallsbeispielen mit `brute_force`: Gleicher kleinster Gewichtsunterschied, und wenn es keine zulässige Verteilung gibt, liefern beide `None`. Dabei kommen auch Fälle vor, in denen die Versicherung *keine* Verteilung zulässt (zum Beispiel zwei Koffer mit sehr verschiedenem Wert).

In [ ]:
import random

zufall = random.Random(5)
keine_loesung = 0
for versuch in range(300):
    anzahl = zufall.randint(2, 11)
    probe_koffer = []
    for _ in range(anzahl):
        probe_koffer.append((zufall.randint(1, 8) * 5000, zufall.randint(1, 30)))
    erwartet = brute_force(probe_koffer)[2]
    ergebnis = suche(probe_koffer)[1]
    if erwartet is None:
        keine_loesung = keine_loesung + 1
    if erwartet != ergebnis:
        print('Fehler bei', probe_koffer)
print('300 Beispiele verglichen, davon', keine_loesung, 'ohne zulässige Verteilung')
print(suche([(50000, 5), (20000, 7)]))

300 Beispiele verglichen, davon 26 ohne zulässige Verteilung
([(50000, 5), (20000, 7)], None, None, 2)


Bei allen 300 Beispielen stimmen die beiden Verfahren überein (es erscheint keine „Fehler“-Zeile). Im letzten Aufruf sieht man einen Fall ohne Lösung: Zwei Koffer mit den Werten 50 000 und 20 000 lassen sich nur mit einem Wertunterschied von 30 000 verteilen. `suche` meldet dann den Gewichtsunterschied `None`.

**Was passiert im schlimmsten Fall?** Die Lösungshinweise weisen darauf hin: Ist der Gesamtwert aller Koffer höchstens 10 000, spielt die Versicherung keine Rolle mehr, und Schranke 1 schneidet nie. Dann bleibt nur die Gewichtsschranke. Das reine Gewichtsproblem ist das berühmte **Partitionsproblem** („teile Zahlen in zwei Gruppen mit möglichst gleicher Summe“), das als NP-vollständig gilt. Ein Programm, das *jede* Eingabe schnell löst, gibt es nur, wenn P = NP wäre. Unsere Beispiele sind nicht von dieser Art, deshalb reicht Branch and Bound.

## Für Neugierige: Zustände zusammenfassen und zählen

Zwei verschiedene Teilverteilungen der ersten `i` Koffer mit **demselben** Wertunterschied und **demselben** Gewichtsunterschied sind für den Rest völlig gleichwertig. Wir müssen den Rest nur einmal untersuchen. Das ist die Idee der **dynamischen Programmierung**:

- Wir halten ein Wörterbuch `zustaende`, das jedem Paar `(dwert, dgew)` die **Anzahl** der Teilverteilungen zuordnet, die es erreichen.
- Für jeden weiteren Koffer wird jeder Zustand zu zwei neuen (Koffer links oder rechts), gleiche Paare werden zusammengezählt.
- Zustände, bei denen `|dwert| − Restwert > 10 000` gilt, werfen wir sofort weg (Schranke 1).

Am Ende stehen im Wörterbuch alle Paare der **zulässigen** Verteilungen mit ihrer Anzahl. So bekommen wir die Zahlen der Tabelle der Lösungshinweise (Verteilungen, Lösungen, optimale Lösungen), ohne jede Verteilung einzeln anzusehen.

In [ ]:
def zaehle(koffer):
    koffer = sorted(koffer, reverse=True)
    n = len(koffer)
    rest_wert = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        rest_wert[i] = rest_wert[i + 1] + koffer[i][0]
    zustaende = {(koffer[0][0], koffer[0][1]): 1}       # Koffer 0 links
    groesste = 1
    for i in range(1, n):
        wert, gewicht = koffer[i]
        neu = {}
        for (dwert, dgew), anzahl in zustaende.items():
            for (a, b) in [(dwert + wert, dgew + gewicht), (dwert - wert, dgew - gewicht)]:
                if abs(a) - rest_wert[i + 1] <= 10000:
                    neu[(a, b)] = neu.get((a, b), 0) + anzahl
        zustaende = neu
        groesste = max(groesste, len(zustaende))
    loesungen = 0
    beste = None
    for (dwert, dgew), anzahl in zustaende.items():
        if abs(dwert) <= 10000:
            loesungen = loesungen + anzahl
            if beste is None or abs(dgew) < beste:
                beste = abs(dgew)
    optimal = 0
    for (dwert, dgew), anzahl in zustaende.items():
        if abs(dwert) <= 10000 and abs(dgew) == beste:
            optimal = optimal + anzahl
    return 2 ** (n - 1), loesungen, beste, optimal, groesste


for anzahl in [15, 20, 30, 40]:
    koffer = lies_koffer('beispieldaten/eingabe' + str(anzahl) + '.txt')
    verteilungen, loesungen, beste, optimal, groesste = zaehle(koffer)
    print('eingabe' + str(anzahl) + ':', verteilungen, 'Verteilungen,', loesungen, 'Lösungen, Gewichtsunterschied', beste,
          ',', optimal, 'optimal; größtes Wörterbuch', groesste, 'Zustände')

eingabe15: 16384 Verteilungen, 77 Lösungen, Gewichtsunterschied 5 , 3 optimal; größtes Wörterbuch 1503 Zustände
eingabe20: 524288 Verteilungen, 1634 Lösungen, Gewichtsunterschied 11 , 5 optimal; größtes Wörterbuch 5095 Zustände
eingabe30: 536870912 Verteilungen, 599375 Lösungen, Gewichtsunterschied 112 , 1 optimal; größtes Wörterbuch 24669 Zustände
eingabe40: 549755813888 Verteilungen, 4144529442 Lösungen, Gewichtsunterschied 5 , 1 optimal; größtes Wörterbuch 126763 Zustände


**Vergleich mit den Lösungshinweisen.** Bei 15, 30 und 40 Koffern stimmt alles überein (16 384 Verteilungen mit 77 Lösungen und 3 optimalen; 2^29 ≈ 537 Millionen Verteilungen mit 599 375 Lösungen und genau 1 optimalen; bei 40 Koffern eine Lösung des Gewichtsunterschieds 5, die Hinweise geben „mehr als eine Million Lösungen“ an, wir zählen über vier Milliarden).

**Bei 20 Koffern weichen wir ab:** Die Hinweise nennen 1 323 Lösungen und 2 optimale, wir zählen **1 634 Lösungen und 5 optimale** (Gewichtsunterschied 11 in beiden Fällen). Wir haben das nachgerechnet: Das Brute-Force-Verfahren aus Schritt 3, auf `eingabe20.txt` angewandt (524 288 Verteilungen), liefert dieselben Zahlen 1 634 und 5 wie die Zählung hier. Warum die Tabelle der Hinweise andere Werte hat, wissen wir nicht (vielleicht ein Zählfehler in der Tabelle). Für die Aufgabe ist es ohne Bedeutung, denn der gesuchte kleinste Unterschied stimmt.

Das größte Wörterbuch hat bei 40 Koffern nur etwa 127 000 Zustände, statt dass man Milliarden Verteilungen einzeln untersucht. Das Verfahren funktioniert, solange die Zahl verschiedener Unterschiede klein bleibt. Hier sind die Werte Vielfache von 1 000 und die Gewichte ganze Zahlen bis höchstens 3 512, deshalb treffen viele Verteilungen auf dasselbe Paar. Bei sehr großen, krummen Zahlen gäbe es kaum gleiche Paare, und das Verfahren hätte keinen Vorteil.

## Alle Beispiele auf einen Blick

| Datei | Gewichtsunterschied | Aufrufe von Branch and Bound | Lösungen | optimale Lösungen |
|---|---|---|---|---|
| `eingabe15.txt` | 5 | 10 852 | 77 | 3 |
| `eingabe20.txt` | 11 | 93 302 | 1 634 | 5 |
| `eingabe30.txt` | 112 | 1 126 434 | 599 375 | 1 |
| `eingabe40.txt` | 5 | 7 508 348 | 4 144 529 442 | 1 |

Die Verteilungen mit Wert- und Gewichtssummen für jede Datei hat die Zelle in Schritt 4 ausgedruckt.

## Zum Schluss: Geht das auch mit MiniZinc?

Ja, und das Modell ist sehr kurz. Das Problem ist ein typisches **ganzzahliges Optimierungsproblem** (die Hinweise erwähnen die lineare Optimierung mit GLPK): 0-1-Entscheidungen, eine Nebenbedingung, ein Ziel.

**Die Variablen:** `rechts[i]` ist 0 oder 1 (Koffer links oder rechts). Den Unterschied links minus rechts schreiben wir als Summe, in der ein Koffer links `+` zählt und einer rechts `−`: `2 * rechts[i] - 1` ist `-1` für links (`rechts[i] = 0`) und `+1` für rechts. Das Vorzeichen ist egal, weil wir den Betrag `abs(...)` nehmen.

**Die Bedingung:** `abs(Wertunterschied) <= 10000`.

**Das Ziel:** `solve minimize dgew;` (der Betrag des Gewichtsunterschieds). Dass der erste Koffer links liegt, ist wie in Python die Symmetrie-Vereinfachung.

Die Daten (`.dzn`) erzeugt eine Python-Zelle aus den eingelesenen Koffern. Für `eingabe15.txt` sieht das so aus:

```minizinc
n = 15;
wert = [96000, 126000, 115000, 125000, 123000, 123000, 112000, 111000, 110000, 110000, 120000, 98000, 130000, 87000, 97000];
gewicht = [27, 21, 27, 15, 19, 46, 47, 32, 14, 20, 50, 19, 22, 50, 46];
```

```minizinc
int: n;                                 % Anzahl der Koffer
array[1..n] of int: wert;
array[1..n] of int: gewicht;

array[1..n] of var 0..1: rechts;        % rechts[i] = 1: Koffer i kommt in den rechten Kofferraum
constraint rechts[1] = 0;               % Symmetrie: Koffer 1 liegt links

% Unterschied links minus rechts: Koffer rechts zaehlt negativ
var int: dgew = abs(sum(i in 1..n)(gewicht[i] * (2 * rechts[i] - 1)));
constraint abs(sum(i in 1..n)(wert[i] * (2 * rechts[i] - 1))) <= 10000;

solve minimize dgew;
output ["dgew = \(dgew)\n", "rechts = \(rechts)\n"];
```

**Ausgabe** für `eingabe15.txt` (Gecode, letzte Lösung, das Optimum):

```
dgew = 5
rechts = [0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0]
```

Die Zeile `==========` am Ende der MiniZinc-Ausgabe bedeutet: Der Solver hat bewiesen, dass es nichts Besseres gibt.

**Messung bei allen vier Beispielen** mit Gecode, Chuffed und CP-SAT. Alle drei beweisen das Optimum und kommen auf dieselben Gewichtsunterschiede wie Python:

| Datei | Python | MiniZinc (Gecode, Chuffed, CP-SAT) |
|---|---|---|
| `eingabe15.txt` | 5 | 5, bewiesen |
| `eingabe20.txt` | 11 | 11, bewiesen |
| `eingabe30.txt` | 112 | 112, bewiesen |
| `eingabe40.txt` | 5 | 5, bewiesen |

Wir haben jede gefundene Verteilung mit unserer Funktion `ist_zulaessig` geprüft. Die Verteilung kann sich von der Python-Lösung unterscheiden, wenn es mehrere optimale gibt (bei 15 und 20 Koffern gibt es 3 bzw. 5), der Gewichtsunterschied ist aber immer derselbe.

**Grenzen:** Bei 40 Koffern wurde Chuffed bei unserer Messung deutlich langsamer als Gecode und CP-SAT (mehr als zehnmal so lange; Gecode und CP-SAT brauchten wenige Sekunden, Chuffed etwa eine viertel Minute). Das Modell hat nur 40 Entscheidungen, trotzdem muss der Solver das Optimum erst beweisen, und das kostet je nach Solver unterschiedlich viel. Wächst die Koffer-Zahl noch weiter, würde auch hier das Partitionsproblem zuschlagen. Für unsere Größen ist MiniZinc eine bequeme Alternative: Das Modell besteht aus drei Zeilen, und man muss sich um Schranken und Rekursion nicht kümmern.